# From Jacobi smoothing to a multigrid V-cycle

Four damped Jacobi steps affect a smooth error and a rapidly oscillating error very differently. We measure what remains of each error, apply an exact coarse-grid correction, and use the same two ingredients to precondition a Poisson solve.

In the [previous notebook](../VO_2026_11_25/02_iterative_solvers.ipynb), the slowly varying error survived smoothing. Can one small solve remove most of what remains?

In [ ]:
from ngsolve import *
from ngsolve.krylovspace import CGSolver
from ngsolve.meshes import MakeStructured2DMesh

## A hierarchy of linear finite element spaces

We use continuous $P_1$ elements on triangular meshes of the unit square with zero Dirichlet values. Each uniform refinement halves the mesh size. The stiffness form on every level is $a(u,v)=\int_\Omega \nabla u\cdot\nabla v\,dx$.

The coarsest grid has four subdivisions per side. We register the built-in multigrid preconditioner before assembly so that it keeps the matrices on all three levels. We also save an exact solve on the middle grid for the two-grid experiment.

In [ ]:
mesh = MakeStructured2DMesh(quads=False, nx=4, ny=4)
V = H1(mesh, order=1, dirichlet=".*", autoupdate=True)
u, v = V.TnT()
a = BilinearForm(grad(u) * grad(v) * dx, symmetric=True)
mg = Preconditioner(a, "multigrid", smoother="point", smoothingsteps=1,
                    cycle=1, inverse="sparsecholesky")
a.Assemble()

mesh.Refine()
a.Assemble()
A_H_inv = a.mat.Inverse(V.FreeDofs(), inverse="sparsecholesky")

mesh.Refine()
a.Assemble()
A = a.mat

## Which errors does each correction remove?

Let $E=U_\ast-U$ be the algebraic error on the fine grid. Its residual is $r=A_h E$. We first update $E$ with damped Jacobi, then use the residual of the smoothed error for one coarse correction.

$$
\begin{align*}
E &\leftarrow E-\tfrac{2}{3}D_h^{-1}A_hE,
& D_h&=\operatorname{diag}(A_h),\\
E &\leftarrow E-P A_H^{-1}P^\top A_hE,
& A_H&=P^\top A_h P.
\end{align*}
$$

NGSolve's nodal prolongation $P$ interpolates a coarse function onto the fine grid. Its transpose restricts a residual by testing it against the coarse basis functions. The stiffness matrix assembled on the middle grid is the Galerkin matrix $A_H$ because the spaces are nested and use the same form.

In [ ]:
D_inv = A.CreateSmoother(V.FreeDofs())  # inverse diagonal on free nodes
P = V.Prolongation().Operator(mesh.levels - 1)
coarse_correction = P @ A_H_inv @ P.T

def energy_norm(E):
    return sqrt(InnerProduct(E, A * E))

Prescribe the nodal values of $\sin(j\pi x)\sin(j\pi y)$ as the initial error. The table measures the remaining fraction of its energy norm $\|E\|_{A_h}=(E^\top A_hE)^{1/2}$. Compare the smooth mode $j=1$ with the oscillatory mode $j=13$.

In [ ]:
print("mode   after 4 Jacobi steps   after coarse correction")
for j in (1, 13):
    error = GridFunction(V)
    error.Interpolate(sin(j * pi * x) * sin(j * pi * y))
    E = error.vec
    initial_norm = energy_norm(E)

    for step in range(4):
        E.data -= (2/3) * (D_inv * (A * E))
    after_smoothing = energy_norm(E) / initial_norm

    E.data -= coarse_correction * (A * E)
    after_correction = energy_norm(E) / initial_norm
    print(f"{j:4d}   {after_smoothing:20.4f}   {after_correction:23.4f}")

Most of the smooth mode survives the four Jacobi steps and is then reduced by coarse correction. For the highly oscillatory mode, the Jacobi steps already account for nearly all of the reduction.

## An additive two-grid preconditioner and a V-cycle

In the **additive** method, both corrections act on the same incoming residual $r$.

$$
\begin{align*}
C_{\mathrm{TG}}^{-1}r &= D_h^{-1}r + P A_H^{-1}P^\top r.
\end{align*}
$$

The built-in **V-cycle** visits successively coarser grids. It applies a forward Gauss–Seidel sweep, restricts the updated residual for the coarse correction, and applies a backward sweep on the way back. The coarsest grid is solved by sparse Cholesky. Both inverse actions are symmetric positive definite on the free space and can be passed to PCG.

In [ ]:
two_grid = D_inv + coarse_correction

source = 2 * (x * (1 - x) + y * (1 - y))
load = LinearForm(source * v * dx).Assemble()
free = Projector(V.FreeDofs(), True)

We now use the chapter's load $f=2[x(1-x)+y(1-y)]$ and start each solve at zero. NGSolve's CG solver monitors the preconditioned residual norm $(r^\top C^{-1}r)^{1/2}$. After each solve, we recompute $\|b-A_h U\|_2/\|b\|_2$ on the free degrees of freedom.

In [ ]:
for name, pre in [("Additive two-grid", two_grid), ("Multigrid V-cycle", mg.mat)]:
    solver = CGSolver(A, pre=pre, tol=1e-10, )
    U = solver.Solve(rhs=load.vec)
    residual = free * (load.vec - A * U)
    print(f"{name}: relative residual {Norm(residual) / Norm(free * load.vec):.2e}")